# PyTorch from Zero

### Main idea

A neural network learns by repeating four steps:

1. **Forward:** make a prediction.
2. **Loss:** measure how wrong it is.
3. **Backward:** calculate gradients.
4. **Update:** move parameters in a direction that reduces the loss.


No dataset download is required. Everything runs on CPU, GPU.

By Raoof Zare Moayedi

# Part A — Setup and tensors

In [ ]:
import math
import random
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

torch.set_num_threads(1)

SEED = 7
random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("PyTorch version:", torch.__version__)
print("Device:", device)

## 1. What is a tensor?

A tensor is an array of numbers.

- A scalar has shape `()`.
- A vector has shape `(n,)`.
- A matrix has shape `(rows, columns)`.
- Images are often stored as `(channels, height, width)`.
- A batch of images is often `(batch, channels, height, width)`.

The **shape** tells us how the numbers are organized.

In [ ]:
scalar = torch.tensor(5.0)
vector = torch.tensor([1.0, 2.0, 3.0])
matrix = torch.tensor([[1.0, 2.0, 3.0],
                       [4.0, 5.0, 6.0]])

print("scalar:", scalar, "shape =", scalar.shape)
print("vector:", vector, "shape =", vector.shape)
print("matrix:\n", matrix, "shape =", matrix.shape)
print("dtype:", matrix.dtype)

### Useful constructors

In [ ]:
zeros = torch.zeros(2, 3)
ones = torch.ones(2, 3)
random_uniform = torch.rand(2, 3)
random_normal = torch.randn(2, 3)
integers = torch.arange(0, 12).reshape(3, 4)

print("zeros:\n", zeros)
print("random normal:\n", random_normal)
print("reshaped integers:\n", integers)

## 2. Indexing and reshaping

In [ ]:
x = torch.arange(12).reshape(3, 4)

print("x:\n", x)
print("First row:", x[0])
print("Second column:", x[:, 1])
print("Bottom-right element:", x[-1, -1])
print("Flattened:", x.flatten())
print("Transposed:\n", x.T)

## 3. Elementwise operations versus matrix multiplication

For tensors of the same shape:

\[
A * B
\]

multiplies matching entries.

Matrix multiplication uses:

\[
A @ B
\]

In [ ]:
A = torch.tensor([[1.0, 2.0],
                  [3.0, 4.0]])
B = torch.tensor([[5.0, 6.0],
                  [7.0, 8.0]])

print("Elementwise A * B:\n", A * B)
print("Matrix product A @ B:\n", A @ B)

## 4. Broadcasting

Broadcasting lets PyTorch combine tensors with compatible shapes.  
A common neural-network example is adding one bias value to every row in a batch.

In [ ]:
batch = torch.tensor([[1.0, 2.0, 3.0],
                      [4.0, 5.0, 6.0]])
bias = torch.tensor([10.0, 20.0, 30.0])

print("batch shape:", batch.shape)
print("bias shape:", bias.shape)
print("batch + bias:\n", batch + bias)

### Checkpoint exercise

Create a matrix with shape `(4, 3)`. Add a vector of shape `(3,)` to it.  
Then calculate the sum of each row.

In [ ]:
# TODO: 

# Part B — Gradients and automatic differentiation

## 5. What does a gradient mean?

For one variable, the derivative is the local slope:

$$
f'(x)=\lim_{\varepsilon\to 0}
\frac{f(x+\varepsilon)-f(x)}{\varepsilon}.
$$

For many variables, the gradient collects all partial derivatives:

$$
\nabla f =
\left[
\frac{\partial f}{\partial x_1},
\frac{\partial f}{\partial x_2},
\ldots
\right].
$$

The gradient points in the direction of the fastest local increase.  
Therefore, moving in the **negative gradient direction** tends to decrease the function.

## 6. First gradient with `backward()`

Consider:

$$
y=x^2+3x+1.
$$

Its derivative is:

$$
\frac{dy}{dx}=2x+3.
$$

At \(x=2\), the derivative should be \(7\).

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x**2 + 3*x + 1

print("y =", y.item())
print("Before backward, x.grad =", x.grad)

y.backward()

print("After backward, x.grad =", x.grad)

### The essential pattern

```python
x = torch.tensor(..., requires_grad=True)
y = some_computation(x)
y.backward()
print(x.grad)
```

- `requires_grad=True` asks PyTorch to record operations involving the tensor.
- `.backward()` applies the chain rule.
- `.grad` stores the resulting derivative for a **leaf tensor**.

## 7. Partial derivatives

Let:

$$
f(x,y)=x^2y+3y.
$$

Then:

$$
\frac{\partial f}{\partial x}=2xy,
\qquad
\frac{\partial f}{\partial y}=x^2+3.
$$

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(4.0, requires_grad=True)

f = x**2 * y + 3*y
f.backward()

print("f =", f.item())
print("df/dx =", x.grad.item())
print("df/dy =", y.grad.item())

## 8. The chain rule as a computational graph

Suppose:

$$
a=2x,\qquad b=a+1,\qquad c=b^2.
$$

Then:

$$
\frac{dc}{dx}
=
\frac{dc}{db}
\frac{db}{da}
\frac{da}{dx}.
$$

Autograd records the operations during the forward pass and traverses them backward.

In [ ]:
x = torch.tensor(3.0, requires_grad=True)
a = 2 * x
b = a + 1
c = b**2

print("x =", x.item(), "a =", a.item(), "b =", b.item(), "c =", c.item())
print("c.grad_fn:", c.grad_fn)

c.backward()
print("dc/dx =", x.grad.item())

## 9. Leaf and non-leaf tensors

A tensor created directly by us is usually a **leaf**.  
A tensor produced by another recorded operation is usually a **non-leaf**.

PyTorch normally stores `.grad` only for leaves because model parameters are leaves.
Use `.retain_grad()` when teaching or debugging and you specifically want a non-leaf gradient.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)  # leaf
a = 3 * x                                  # non-leaf
a.retain_grad()
y = a**2

y.backward()

print("x.is_leaf:", x.is_leaf)
print("a.is_leaf:", a.is_leaf)
print("dy/dx:", x.grad.item())
print("dy/da:", a.grad.item())

## 10. `.backward()` versus `torch.autograd.grad`

- `.backward()` **accumulates** derivatives into the `.grad` fields of leaves.
- `torch.autograd.grad(...)` returns derivatives directly.

The second form is convenient when we want a derivative as a value for another calculation.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x**3

(dy_dx,) = torch.autograd.grad(y, x)

print("dy/dx =", dy_dx.item())
print("x.grad is still:", x.grad)

## 11. Higher-order derivatives

For:

$$
y=x^4,
$$

we have:

$$
\frac{dy}{dx}=4x^3,
\qquad
\frac{d^2y}{dx^2}=12x^2.
$$

To differentiate a derivative, tell PyTorch to construct another graph using `create_graph=True`.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x**4

(dy_dx,) = torch.autograd.grad(y, x, create_graph=True)
(d2y_dx2,) = torch.autograd.grad(dy_dx, x)

print("First derivative:", dy_dx.item())
print("Second derivative:", d2y_dx2.item())

## 12. Vector outputs and vector–Jacobian products

Calling `.backward()` directly is simplest when the output is a scalar.

For a vector output, we can:

1. reduce it to a scalar, such as `y.sum().backward()`, or
2. provide a vector of upstream weights to `backward`.

The second operation computes a vector–Jacobian product.

In [ ]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = x**2

# The upstream vector says how strongly each output contributes.
v = torch.tensor([1.0, 0.5, -1.0])
y.backward(gradient=v)

print("y =", y)
print("v =", v)
print("gradient =", x.grad)

## 13. Jacobians

For a vector-valued function:

$$
f:\mathbb{R}^n\rightarrow\mathbb{R}^m,
$$

the Jacobian stores every output derivative with respect to every input.

In [ ]:
def vector_function(z):
    return torch.stack([
        z[0]**2 + z[1],
        z[0] * z[1]
    ])

z = torch.tensor([2.0, 3.0], requires_grad=True)
J = torch.autograd.functional.jacobian(vector_function, z)

print("Jacobian shape:", J.shape)
print(J)

## 14. Gradients accumulate

PyTorch adds new gradients to existing `.grad` values.  
This is useful in some advanced methods, but it is a common beginner bug.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)

y1 = x**2
y1.backward()
print("After first backward:", x.grad.item())   # 4

y2 = 3*x
y2.backward()
print("After second backward:", x.grad.item())  # 4 + 3 = 7

x.grad.zero_()
print("After zeroing:", x.grad.item())

## 15. Turning gradient tracking off

During prediction, we usually do not need to build a gradient graph.

```python
with torch.no_grad():
    prediction = model(x)
```

`detach()` creates a tensor that shares the numerical value but is disconnected from the current graph.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x**2

print("y.requires_grad:", y.requires_grad)
print("y.detach().requires_grad:", y.detach().requires_grad)

with torch.no_grad():
    z = x**3

print("z.requires_grad inside no_grad:", z.requires_grad)

## 16. Check autograd using finite differences

For a small \(\varepsilon\):

$$
f'(x)\approx
\frac{f(x+\varepsilon)-f(x-\varepsilon)}{2\varepsilon}.
$$

This is slower and less accurate than autograd, but useful as a sanity check.

In [ ]:
def f_number(t):
    return t**3 + 2*t

x_value = 1.7
epsilon = 1e-4

finite_difference = (
    f_number(x_value + epsilon) - f_number(x_value - epsilon)
) / (2 * epsilon)

x = torch.tensor(x_value, requires_grad=True)
f = x**3 + 2*x
f.backward()
autograd_value = x.grad.item()

print("Finite difference:", finite_difference)
print("Autograd:", autograd_value)
print("Absolute difference:", abs(finite_difference - autograd_value))

### Gradient checkpoint exercise

For

$$
f(x,y)=(xy+2)^3,
$$

calculate both partial derivatives at \(x=2\), \(y=3\).

1. First predict the result by hand.
2. Then verify it with PyTorch.

In [ ]:
# TODO: Write your solution here.\n# Start by creating the required tensors with requires_grad=True when needed.\n

# Part C — Gradient descent by hand

## 17. Learning a line

We generate points approximately following:

$$
y=3x-2.
$$

Our model is:

$$
\hat y=wx+b.
$$

The mean squared error is:

$$
L(w,b)=\frac{1}{N}\sum_i(\hat y_i-y_i)^2.
$$

Gradient descent updates:

$$
w \leftarrow w-\eta\frac{\partial L}{\partial w},
\qquad
b \leftarrow b-\eta\frac{\partial L}{\partial b}.
$$

In [ ]:
torch.manual_seed(SEED)

x_data = torch.linspace(-2, 2, 80)
noise = 0.35 * torch.randn_like(x_data)
y_data = 3*x_data - 2 + noise

plt.scatter(x_data, y_data, s=18)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Synthetic data")
plt.show()

In [ ]:
w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

learning_rate = 0.05
loss_history_manual = []

for step in range(120):
    prediction = w*x_data + b
    loss = ((prediction - y_data)**2).mean()

    loss.backward()

    with torch.no_grad():
        w -= learning_rate * w.grad
        b -= learning_rate * b.grad

    # Gradients accumulate, so clear them after the update.
    w.grad.zero_()
    b.grad.zero_()

    loss_history_manual.append(loss.item())

    if step % 20 == 0:
        print(
            f"step={step:3d}  loss={loss.item():.4f}  "
            f"w={w.item():.3f}  b={b.item():.3f}"
        )

In [ ]:
plt.plot(loss_history_manual)
plt.xlabel("Training step")
plt.ylabel("Mean squared error")
plt.title("Manual gradient descent")
plt.show()

with torch.no_grad():
    fitted_line = w*x_data + b

plt.scatter(x_data, y_data, s=18, label="data")
plt.plot(x_data, fitted_line, label="learned line")
plt.legend()
plt.show()

print("Learned w:", w.item())
print("Learned b:", b.item())

### What just happened?

The learning loop contained the full structure used for large neural networks:

```python
prediction = model(inputs)   # forward
loss = loss_fn(prediction, targets)
loss.backward()              # gradients
update_parameters()
clear_gradients()
```

# Part D — PyTorch building blocks

## 18. `nn.Parameter`

A model parameter is a tensor that should be learned.  
`nn.Parameter` automatically defaults to `requires_grad=True`.

In [ ]:
weight = nn.Parameter(torch.tensor(2.0))
print(weight)
print("requires_grad:", weight.requires_grad)

## 19. `nn.Linear`

A linear layer computes:

$$
Y=XW^\top+b.
$$

For `nn.Linear(in_features=3, out_features=2)`:

- the weight shape is `(2, 3)`;
- the bias shape is `(2,)`.

In [ ]:
layer = nn.Linear(in_features=3, out_features=2)

sample_batch = torch.randn(5, 3)
output = layer(sample_batch)

print("Input shape:", sample_batch.shape)
print("Weight shape:", layer.weight.shape)
print("Bias shape:", layer.bias.shape)
print("Output shape:", output.shape)

## 20. Build an MLP with `nn.Module`

`nn.Module` is the base class for models and layers.

A model usually defines:

- layers in `__init__`;
- the forward computation in `forward`.

In [ ]:
class TinyMLP(nn.Module):
    def __init__(self, hidden_size=16):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(2, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, 1),
        )

    def forward(self, x):
        # Remove the final singleton dimension: (batch, 1) -> (batch,)
        return self.network(x).squeeze(-1)

model = TinyMLP(hidden_size=16).to(device)
print(model)

number_of_parameters = sum(p.numel() for p in model.parameters())
print("Trainable parameter count:", number_of_parameters)

## 21. Common loss functions

| Task | Output of model | Target | Loss |
|---|---|---|---|
| Regression | real number(s) | real number(s) | `nn.MSELoss()` |
| Binary classification | one raw logit | 0 or 1 | `nn.BCEWithLogitsLoss()` |
| Multi-class classification | one raw logit per class | integer class index | `nn.CrossEntropyLoss()` |

**Important:** `BCEWithLogitsLoss` already includes the sigmoid operation.  
`CrossEntropyLoss` already includes the softmax-like normalization required for the loss. Pass raw logits.

In [ ]:
# Tiny shape demonstration for multi-class classification
multiclass_logits = torch.tensor([
    [2.0, 0.5, -1.0],
    [-0.2, 0.1, 1.4],
])
multiclass_targets = torch.tensor([0, 2])

ce_loss = nn.CrossEntropyLoss()
print("Cross-entropy loss:", ce_loss(multiclass_logits, multiclass_targets).item())

## 22. Optimizers

An optimizer updates model parameters using their gradients.

Typical pattern:

```python
optimizer.zero_grad()
loss.backward()
optimizer.step()
```

In this notebook we use Adam for the final classifier. SGD is conceptually closer to the manual update shown earlier.

## Final cheat sheet

```python
# Create a differentiable tensor
x = torch.tensor(2.0, requires_grad=True)

# Build a computation
y = x**2

# Calculate dy/dx
y.backward()
print(x.grad)

# Standard training step
optimizer.zero_grad(set_to_none=True)
logits = model(inputs)
loss = loss_fn(logits, targets)
loss.backward()
optimizer.step()

# Evaluation
model.eval()
with torch.no_grad():
    predictions = model(inputs)
```

### The one sentence to remember

**Autograd records differentiable tensor operations during the forward pass and applies the chain rule backward to calculate gradients.**